### 1️⃣ IMPORTACIONES Y VARIABLES INICIALES

In [ ]:
# Importa funciones para manipular columnas y tipos de datos.
from pyspark.sql.functions import col, round
from pyspark.sql import functions as F  # Alias para acceder a pyspark.sql.functions como F.col, F.max, etc.
from pyspark.sql.types import DecimalType, DoubleType, FloatType
from delta.tables import DeltaTable
from pyspark.sql.utils import AnalysisException
from py4j.protocol import Py4JJavaError
import re

# Ruta donde se guardarán las tablas en capa Silver
ruta_silver = 'abfss://DEV@onelake.dfs.fabric.microsoft.com/LH_SILVER_DEV.Lakehouse/Tables'

# Variables de control para decidir qué secciones de código ejecutar.
# [Nota] Sería más seguro iniciarlas en 0 (int) o False (bool) para evitar confusiones en las comparaciones.
ax = ''  
dataflow = ''
dynamics = ''

# Mostrar el valor actual de las variables de control.
print(ax)
print(dataflow) 
print(dynamics) 


### 2️⃣ FUNCIONES DE LIMPIEZA DE DATOS

#### limpiar_nombres_columnas(df)

In [ ]:
def limpiar_nombres_columnas(df):
    """
    Limpia y estandariza nombres de columnas para evitar problemas.
    - Quita saltos de línea y tabulaciones.
    - Elimina espacios al inicio y fin.
    - Sustituye espacios múltiples por guiones bajos (_).
    """    
    columnas_limpias = []

    for col in df.columns:
        nuevo_nombre = col
        nuevo_nombre = re.sub(r'\n', ' ', nuevo_nombre)  # Reemplaza saltos de línea por espacio
        nuevo_nombre = re.sub(r'\t', ' ', nuevo_nombre)  # Reemplaza tabulaciones por espacio
        nuevo_nombre = nuevo_nombre.strip()  # Quita espacios al inicio y final
        nuevo_nombre = re.sub(r'\s+', '_', nuevo_nombre)  # Reemplaza varios espacios por guion bajo
        columnas_limpias.append(nuevo_nombre)

    # Renombra columnas originales con las limpias.
    for old, new in zip(df.columns, columnas_limpias):
        df = df.withColumnRenamed(old, new)

    return df

#### eliminar_prefijo_mserp(df)

In [ ]:
def eliminar_prefijo_mserp(df):
    """
    Elimina el prefijo 'mserp_' de los nombres de columnas.
    """
    # Usa alias para renombrar columnas en un solo paso.
    df = df.select([F.col(columna).alias(columna.replace('mserp_', '')) for columna in df.columns])
    
    return df

#### round_decimal_columns(df, precision=4)

In [ ]:
def round_decimal_columns(df, precision=4):
    """
    Redondea todas las columnas numéricas decimales a la precisión indicada.
    """
    for field in df.schema.fields:
        if isinstance(field.dataType, (DecimalType, DoubleType)):
            df = df.withColumn(field.name, round(col(field.name), precision))
    return df

#### eliminar_filas_nulas(df)

In [ ]:
def eliminar_filas_nulas(df):
    """
    Elimina filas que son completamente nulas.
    """
    return df.dropna(how='all')

#### limpiar_decimales_enteros(df)

In [ ]:
def limpiar_decimales_enteros(df): 
    """
    Convierte columnas numéricas a string si todos sus valores son enteros (sin decimales).
    """
    for field in df.schema.fields:
        if isinstance(field.dataType, (DecimalType, DoubleType, FloatType)):
            col_name = field.name

            # Calcula la parte fraccionaria de la columna (% 1 da el decimal).
            fraccion = df.select((col(col_name) % 1).alias("fraccion"))
            # [Nota / Mejora] Esto usa collect() y rompe paralelismo si el DF es grande, usar First o Last.
            max_frac = fraccion.agg({"fraccion": "max"}).collect()[0][0]

            # Si no hay parte fraccionaria en ninguna fila → convertir a entero y luego a string.
            if max_frac == 0:
                df = df.withColumn(
                    col_name,
                    col(col_name).cast("long").cast("string")
                )
    return df

### 3️⃣ PROCESO DE CARGA Y GUARDADO DE TABLAS

In [ ]:
esquema = 'dim'  # Subcarpeta donde se guardarán las tablas procesadas.

def procesar_y_guardar_tabla(nombre_tabla):
    
    """
    Procesa un DataFrame:
    - Limpia nombres de columnas.
    - Redondea columnas decimales.
    - Elimina filas completamente nulas.
    - Guarda en formato Delta en capa Silver.
    """
    
    # Extrae nombre base quitando el prefijo del esquema.
    nombre_base = nombre_tabla.split(".")[-1]
    
    # Obtiene el DataFrame desde un diccionario global.
    df = dataframes[nombre_tabla]
    df = limpiar_nombres_columnas(df)
    df = round_decimal_columns(df)
    df = eliminar_filas_nulas(df)

    display(nombre_base)  # Muestra el nombre de la tabla para trazabilidad.

    # Guarda el DF en Delta, sobrescribiendo datos y esquema.
    ruta_destino = f"{ruta_silver}/{esquema}/{nombre_base}"
    df.write.format("delta")\
        .mode("overwrite")\
        .option("overwriteSchema", "true")\
        .save(ruta_destino)

### 4️⃣ EJECUCIÓN CONDICIONAL PARA DATAFLOW

In [ ]:
# Solo se ejecuta si la variable de control vale 1.
if dataflow == 1:
    
    # Lista de tablas de la capa Bronze a cargar.
    tablas = [
        "LH_BRONZE_DEV.dbo.PPTO",
        "LH_BRONZE_DEV.dbo.SMMBUSRELCHAIN",
        "LH_BRONZE_DEV.dbo.PuestoActual",
        "LH_BRONZE_DEV.dbo.Clientes",
    ]

    # Diccionario para guardar DataFrames cargados.
    dataframes = {}
    for tabla in tablas:
        consulta = f"SELECT * FROM {tabla}"
        df = spark.sql(consulta)
        dataframes[tabla] = df

    # Procesa y guarda cada tabla.
    for tabla in tablas:
        procesar_y_guardar_tabla(tabla)

### 5️⃣ FUNCIONES PARA MERGE / UPSERT CON DELTA

#### existe_tabla_delta(ruta_tabla: str) -> bool

In [ ]:
def existe_tabla_delta(ruta_tabla: str) -> bool:
    
    """
    Verifica si existe una tabla Delta en la ruta indicada.
    """
    
    try:
        return DeltaTable.isDeltaTable(spark, ruta_tabla)
    except:
        return False

#### upsert_dataframes_merge(df_nuevo, clave_unica: str, ruta_tabla: str)

In [ ]:
def upsert_dataframes_merge(df_nuevo, clave_unica: str, ruta_tabla: str):
    
    """
    Realiza un MERGE (upsert) en Delta:
    - Si la clave existe, actualiza.
    - Si no, inserta.
    """
    
    delta_table = DeltaTable.forPath(spark, ruta_tabla)
    df_actual = delta_table.toDF()

    # Contadores para diagnóstico (filas antes, nuevas, después).
    cantidad_inicial = df_actual.count()
    cantidad_nuevo = df_nuevo.count()

    columnas = df_nuevo.columns

    # Ejecuta MERGE
    delta_table.alias("actual").merge(
        source=df_nuevo.alias("nuevo"),
        condition=f"actual.{clave_unica} = nuevo.{clave_unica}"
    ).whenMatchedUpdate(set={c: f"nuevo.{c}" for c in columnas}) \
     .whenNotMatchedInsert(values={c: f"nuevo.{c}" for c in columnas}) \
     .execute()

    df_final = spark.read.format("delta").load(ruta_tabla)
    cantidad_final = df_final.count()

    return cantidad_inicial, cantidad_nuevo, cantidad_final

### 6️⃣ MAIN - LECTURA DE PARQUET + UPSERT A DELTA (FASE DYNAMICS)

In [ ]:
# Excepciones comunes al trabajar con Spark/Java desde PySpark.
def procesar_archivos_parquet_upsert(
    archivos,
    nombres_tablas,
    ruta_silver,
    eliminar_prefijo_fn,
    limpiar_columnas_fn,
    redondear_columnas_fn,
    limpiar_decimales_enteros_fn,
    claves_unicas_por_tabla
):
    """
    [Explicación sencilla]
    Lee varios archivos Parquet (uno por tabla), aplica transformaciones de limpieza,
    y escribe en Delta (capa Silver). Si la tabla ya existe, hace MERGE (upsert);
    si no existe, crea la tabla.

    [Detalle técnico]
    - Recibe funciones como parámetros (first-class functions) para usar una misma
      tubería con diferentes transformaciones.
    - Usa la API Java de Hadoop FileSystem desde el gateway Py4J para verificar si
      el archivo existe y su tamaño.
    """
    
    # Acceso a clases Java de Hadoop vía Py4J (Spark JVM).
    hadoop_fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
    Path = spark._jvm.org.apache.hadoop.fs.Path

    # Recorre dos listas en paralelo: cada archivo con su nombre lógico de tabla.
    for archivo, nombre_tabla in zip(archivos, nombres_tablas):
        print(f"\n▶ Procesando tabla: {nombre_tabla}")
        
        try:
            # Verifica que el archivo existe y no está vacío.
            # Si el path apunta a un folder, getFileStatus obtiene el status del folder; si es un archivo único, valida tamaño.
            file_status = hadoop_fs.getFileStatus(Path(archivo))
            file_length = file_status.getLen()

            if file_length == 0:
                print(f"⚠ El archivo '{archivo}' está vacío (0 bytes). Se omite.")
                continue

            # Carga el Parquet como DataFrame.
            # [Nota/Mejora] Si se conoce el esquema, definirlo explícitamente evita inferencia y acelera (y asegura tipos estables).
            df_nuevo = spark.read.parquet(archivo)

        except Py4JJavaError as e:
            # Error del lado JVM (Java). Se muestra el mensaje de la excepción Java.
            print(f"❌ Error leyendo el archivo '{archivo}': {e.java_exception.getMessage()}")
            print("⚠ Se omite esta tabla y se continúa con las siguientes.\n")
            continue
        except Exception as e:
            # Cualquier otro error Python.
            print(f"❌ Error inesperado leyendo '{archivo}': {e}")
            print("⚠ Se omite esta tabla y se continúa con las siguientes.\n")
            continue

        # Aplica una serie de transformaciones al DF cargado.
        # Estas funciones se pasaron como parámetros: esto hace el pipeline reutilizable.
        df_nuevo = eliminar_prefijo_fn(df_nuevo)          # quitar 'mserp_' de los nombres de columnas
        df_nuevo = limpiar_columnas_fn(df_nuevo)           # espacios → guion_bajo, etc.
        df_nuevo = redondear_columnas_fn(df_nuevo)         # redondeo a 4 decimales
        df_nuevo = limpiar_decimales_enteros_fn(df_nuevo)  # columnas sin fracción → cast a entero (y string)

        # Define la ruta destino en Silver para esta tabla.
        ruta_tabla = f"{ruta_silver}/fact/{nombre_tabla}"
        
        # Obtiene la clave única para hacer el MERGE.
        # [Nota/Mejora] Si falta la clave en el dict, esto lanzará KeyError; se podría validar con .get() y lanzar un mensaje claro.
        clave_unica = claves_unicas_por_tabla[nombre_tabla]

        # Si la tabla Delta ya existe → MERGE; si no → inserción inicial.
        if existe_tabla_delta(ruta_tabla):
            # Upsert: update cuando coincida clave_unica, insert cuando no.
            upsert_dataframes_merge(
                df_nuevo, clave_unica, ruta_tabla
            )
            print(f"✔ MERGE ejecutado correctamente.")
        else:
            # No hay tabla previa → se crea desde cero.
            # [Nota/Mejora] Considera .option("overwriteSchema","true") si el esquema pudiera variar entre cargas iniciales.
            df_nuevo.write.format("delta").mode("overwrite").save(ruta_tabla)
            print(f"ℹ No existía tabla previa. Se insertaron los datos sin conteo explícito.")

### 7️⃣ LISTAS DE ARCHIVOS Y NOMBRES LÓGICOS (DYNAMICS)

In [ ]:
# Paths relativos dentro de OneLake/Files para los Parquet de Dynamics.
archivos = [
    "Files/DYNAMICS/CUSTINVOICEJOUR.parquet",
    "Files/DYNAMICS/CUSTINVOICETRANS.parquet",
    "Files/DYNAMICS/CUSTTABLE.parquet", 
    "Files/DYNAMICS/SALESTABLE.parquet",
    "Files/DYNAMICS/INVENTTRANSORIGIN.parquet",
    "Files/DYNAMICS/INVENTTRANS.parquet",
    "Files/DYNAMICS/INVENTDIM.parquet"
]

# Nombres lógicos de las tablas (en el mismo orden que 'archivos').
nombres_tablas = [
    "CUSTINVOICEJOUR",
    "CUSTINVOICETRANS",
    "CUSTTABLE",
    "SALESTABLE",
    "INVENTTRANSORIGIN",
    "INVENTTRANS",
    "INVENTDIM"
]

# Claves únicas de cada tabla para el MERGE.
# Estas claves deben ser realmente únicas en origen; si no, MERGE puede actualizar múltiples filas.
claves_unicas_por_tabla = {
    "CUSTINVOICEJOUR": "invoiceid",
    "CUSTINVOICETRANS": "sourcekey",
    "CUSTTABLE": "custtablebientityid",
    "SALESTABLE": "salestablebientityid",
    "INVENTTRANSORIGIN": "inventtransid",
    "INVENTTRANS": "inventtransid",
    "INVENTDIM": "inventdimid"
}

# Ejecuta el pipeline general: leer → limpiar → (merge|insert) → Delta.
procesar_archivos_parquet_upsert(
    archivos,
    nombres_tablas,
    ruta_silver,
    eliminar_prefijo_mserp,
    limpiar_nombres_columnas,
    round_decimal_columns,
    limpiar_decimales_enteros,
    claves_unicas_por_tabla
)

### 8️⃣ CARGAS CONDICIONALES (DYNAMICS = 1)

In [ ]:
# Solo ejecuta este bloque si la bandera 'dynamics' es 1.
if dynamics == 1:
    
    # Lista de archivos Parquet (otra pareja de tablas de Dynamics)
    archivos = [
        "Files/DYNAMICS/DIRPARTYTABLE.parquet",
        "Files/DYNAMICS/HCMWORKER.parquet"
    ]

    # Diccionario para guardar los DataFrames leídos por archivo.
    dataframes = {}

    # Lee cada archivo Parquet y lo guarda en el dict.
    for archivo in archivos:
        df = spark.read.parquet(archivo)  # Leer el archivo Parquet
        dataframes[archivo] = df  # Almacenar el DataFrame con el nombre del archivo como clave

    # Procesa DIRPARTYTABLE: limpiar → redondear → escribir.
    # Acceder a un DataFrame específico
    df_DIRPARTYTABLE = dataframes["Files/DYNAMICS/DIRPARTYTABLE.parquet"]
    # Aplicar las funciones de limpieza
    df_DIRPARTYTABLE = eliminar_prefijo_mserp(df_DIRPARTYTABLE)
    df_DIRPARTYTABLE = limpiar_nombres_columnas(df_DIRPARTYTABLE)
    df_DIRPARTYTABLE = round_decimal_columns(df_DIRPARTYTABLE)
    #Escribir la tabla
    df_DIRPARTYTABLE.write.format("delta").mode("overwrite").save(ruta_silver + '/fact/DIRPARTYTABLE')

    # Procesa HCMWORKER: limpiar → redondear → escribir.
    # Acceder a un DataFrame específico
    df_HCMWORKER = dataframes["Files/DYNAMICS/HCMWORKER.parquet"]
    # Aplicar las funciones de limpieza
    df_HCMWORKER = eliminar_prefijo_mserp(df_HCMWORKER)
    df_HCMWORKER = limpiar_nombres_columnas(df_HCMWORKER)
    df_HCMWORKER = round_decimal_columns(df_HCMWORKER)
    #Escribir la tabla
    df_HCMWORKER.write.format("delta").mode("overwrite").save(ruta_silver + '/fact/HCMWORKER')

### 9️⃣ FUNCIÓN GENÉRICA: PROCESAR Y GUARDAR (USA 'esquema')

In [ ]:
def procesar_y_guardar(nombre_archivo):
    
    """
    Toma el nombre de un archivo parquet ya cargado en 'dataframes',
    limpia columnas, redondea y guarda en Delta bajo {ruta_silver}/{esquema}/{nombre_base}.
    """
    
    # Obtener el nombre base sin ruta ni extensión
    nombre_base = nombre_archivo.split("/")[-1].replace(".parquet", "")
    
    # Cargar el DataFrame correspondiente y limpiar sus columnas
    # Revisar, ya que lee 2 veces 'dataframes[nombre_archivo]', debería leer df en la segunda línea
    # df = round_decimal_columns(df)
    df = limpiar_nombres_columnas(dataframes[nombre_archivo])
    df = round_decimal_columns(dataframes[nombre_archivo])
    print(f"\n▶ Procesando tabla: {nombre_base}")

    # Guardar el DataFrame en formato Delta en la ruta de silver
    df.write.format("delta")\
        .mode("overwrite")\
        .option("overwriteSchema", "true")\
        .save(f"{ruta_silver}/{esquema}/{nombre_base}")

### 🔟 CARGA CONDICIONAL (AX = 1)

In [ ]:
# Condicionar la ejecución del código
if ax == 1:

# LECTURA DE ARCHIVOS Y GUARDADO COMO DATAFRAMES
# Lista de archivos Parquet
    archivos = [
        "Files/AX/CATALOGOS/NUMBERSEQUENCEGROUP.parquet",
        "Files/AX/CATALOGOS/INVENTITEMGROUPITEM.parquet",  # Añadir más archivos si es necesario
        "Files/AX/CATALOGOS/HCMPOSITIONHIERARCHY.parquet",
        "Files/AX/CATALOGOS/HCMPOSITIONDETAIL.parquet",
        "Files/AX/CATALOGOS/HCMPOSITIONDURATION.parquet",  
        "Files/AX/CATALOGOS/HCMPOSITION.parquet"
    ]

    # Diccionario para almacenar DF por archivo.
    dataframes = {}

    # Carga cada parquet en 'dataframes[archivo]'.
    for archivo in archivos:
        # Leer el archivo Parquet
        df = spark.read.parquet(archivo)
        dataframes[archivo] = df

    # Recorre y guarda cada DF usando la función genérica.
    for archivo in archivos:
        procesar_y_guardar(archivo)

    # Caso específico: HCMPOSITIONHIERARCHY requiere un cast a fecha.
    df_HCMPOSITIONHIERARCHY = dataframes["Files/AX/CATALOGOS/HCMPOSITIONHIERARCHY.parquet"]
    df_HCMPOSITIONHIERARCHY = limpiar_nombres_columnas(df_HCMPOSITIONHIERARCHY)
    df_HCMPOSITIONHIERARCHY = round_decimal_columns(df_HCMPOSITIONHIERARCHY)
    # Convierte la columna 'ValidTo' a tipo fecha (si es necesario)
    # 'cast("date")' trunca parte de tiempo. Si la columna viene como string con formato irregular, podría dar nulls.
    df_HCMPOSITIONHIERARCHY = df_HCMPOSITIONHIERARCHY.withColumn("ValidTo", col("ValidTo").cast("date"))
    df_HCMPOSITIONHIERARCHY.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(ruta_silver + '/dim/HCMPOSITIONHIERARCHY')